# 09: ablacje: czy potrzebna jest kamera B? czy potrzebna jest geometria?

Ten sam zbiór `rs_video_x4k`, te same epoki i parametry co modele, z którymi porównujemy. Dwa pytania:

**Epoki:** modele w głównym porównaniu architektur (zwykły U-Net, `ae`, `corr`, `corrgeom`) uczymy **200 epok**.
Ablacje i testy kontrolne porównują pary przy tej samej liczbie epok co istniejący run (30 jak w 06, 80 jak `corr` z 08),
żeby nie powtarzać długich treningów; ważne jest tylko, że w parze liczba epok jest równa.

**A. Kamera B.** Te same modele co w 06 i 08, ale **zamiast obrazu z kamery B sieć dostaje kopię obrazu A** (`--single`).
Architektura i liczba wag bez zmian, sieć nie ma tylko drugiej chwili odczytu. Na etapie 1 (MotionNet, sam ruch kamery):
+8,9 dB z kamerą B, +0,2 dB bez niej (`docs/etap1_motionnet.md`); tu sprawdzamy to na ruchu obiektów.

**B. Geometria („korelacja + geometria”).** Sieć z `corr` (ta sama architektura, 6,2 M wag, uczona od zera) zamiast obrazu
przewiduje **przesunięcie A→B**, a stała geometria RS v1 zamienia je na obraz, dokładnie jak w RAFT + geometria v1.
To szczebel między `corr` (dopasowanie bez geometrii) a RAFT + geometrią (wstępnie wytrenowane dopasowanie + geometria):
* dużo lepiej niż `corr` → kluczem jest geometria,
* blisko `corr` → kluczem jest dobre dopasowanie RAFT (wstępny trening na zbiorach przepływu).

| sekcja | model | `--mode` | porównanie z | epoki |
|---|---|---|---|---|
| 3 | zwykły U-Net bez kamery B | `plain --single` | `*_unet_plain` z 06 (28,45 dB) | 30 |
| 4 | enkoder + korelacja bez kamery B | `corr --single` | `*_corr` z 08 | 80 |
| 5 | korelacja + geometria | `corrgeom` | `*_corr200` (corr, 200 epok) i RAFT + geometria v1 z 04 (32,86 bez uczenia / 34,30 dB) | 200 |

`psnr` liczony jak w 04 (obszar widoczny w A); `psnr_full` = cały obraz (wspólna miara dla wszystkich modeli).
Sekcja 6 (opcjonalna): testy kontrolne współczynnika uczenia i rozmiaru sieci. Sekcje 3–6 są niezależne: można je puścić w osobnych sesjach Colaba (w każdej: SETUP, 1, 2, potem swoja sekcja).
**Przed uruchomieniem zrób commit + push.**

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'kordiusz/RollingShutter'   # repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Zbiór danych (rozpakowanie z Drive na dysk maszyny Colaba)

In [ ]:
import os, glob, shutil, subprocess
DATASET_NAME = 'rs_video_x4k'
z = P.find_dataset_zip(DATASET_NAME)
assert z, f'Brak {DATASET_NAME}.zip w {P.DATASETS} - wrzuć tam spakowany folder data/{DATASET_NAME}'
UNZ = f'/content/unz_{DATASET_NAME}'
DONE = f'{UNZ}/.rozpakowane'          # znacznik: rozpakowanie skończyło się bez błędu
if not os.path.exists(DONE):
    shutil.rmtree(UNZ, ignore_errors=True)            # resztki po przerwanym rozpakowaniu
    LOCAL_ZIP = f'/content/{DATASET_NAME}.zip'
    # najpierw kopia na dysk Colaba: unzip czytający wprost z Drive potrafi się urwać ("zipfile read error")
    print(f'kopiuję {z} ({os.path.getsize(z):,} B) na dysk Colaba ...')
    subprocess.run(['cp', str(z), LOCAL_ZIP], check=True)
    t = subprocess.run(['unzip', '-tq', LOCAL_ZIP], capture_output=True, text=True)
    assert t.returncode == 0, ('Archiwum uszkodzone - wgraj zip na Drive ponownie (lokalnie ma 1 233 434 992 B):\n'
                               + (t.stdout + t.stderr)[-1500:])
    print('archiwum OK, rozpakowuję ...')
    subprocess.run(['unzip', '-q', LOCAL_ZIP, '-d', UNZ], check=True)
    os.remove(LOCAL_ZIP)
    open(DONE, 'w').close()
# folder z podfolderami train/ i val/ (niezależnie od tego, jak spakowano)
DATA = next(os.path.dirname(p) for p in sorted(glob.glob(f'{UNZ}/**/train', recursive=True)))
assert os.path.isdir(f'{DATA}/val'), f'brak {DATA}/val - usuń {UNZ} i uruchom komórkę ponownie'
print('DATA =', DATA, '|', len(os.listdir(f'{DATA}/train')), 'train /', len(os.listdir(f'{DATA}/val')), 'val')
WORKERS = min(8, os.cpu_count())

## 2. Nazwy runów
Przy wznawianiu innego dnia wpisz datę istniejących folderów z `runs/`.

In [ ]:
from datetime import date
from IPython.display import Image, display
DAY = str(date.today())          # np. '2026-11-12' przy wznawianiu
RUNS = {'plain': f'{DAY}_unet_plain_single', 'corr': f'{DAY}_corr_single', 'corrgeom': f'{DAY}_corrgeom'}
EPOCHS = {'plain': 30, 'corr': 80, 'corrgeom': 200}   # ablacje: jak pary z 06 / 08; corrgeom: jak główne porównanie (200)
BS = 16

def train(mode, resume=False, lr=3e-4):
    run = RUNS[mode]
    single = '' if mode == 'corrgeom' else '--single'   # ablacja B tylko dla plain / corr
    res = f'--resume "{P.RUNS / run / "last.pt"}"' if resume else ''
    !python src/train_unet.py --data "{DATA}" --mode {mode} {single} --epochs {EPOCHS[mode]} --bs {BS} --lr {lr} --workers {WORKERS} --out "{P.RUNS}" --name {run} {res}
    prev = sorted((P.RUNS / run).glob('preview_ep*.png'))
    if prev:
        print(prev[-1]); display(Image(str(prev[-1])))
print(RUNS)

## 3. A: zwykły U-Net bez kamery B, 30 epok (~11 min na L4)
Po rozłączeniu: SETUP, 1, 2, potem `train('plain', resume=True)`.

In [ ]:
train('plain')

## 4. A: enkoder + korelacja bez kamery B, 80 epok

In [ ]:
train('corr')

## 5. B: korelacja + geometria, 200 epok (~2–2,5 h na L4)
Porównanie z `corr` uczonym też 200 epok (run `*_corr200`, uruchamiany w kopii notatnika 08: `EPOCHS['corr'] = 200; RUNS['corr'] = f'{DAY}_corr200'; train('corr')`).
Podgląd: RS A | RS B | wynik | błąd ×4 | wzorzec.

In [ ]:
train('corrgeom')

## 6. (opcjonalnie) Testy kontrolne: współczynnik uczenia i rozmiar sieci

Nie strojenie, tylko odpowiedź na dwa pytania, które łatwo zadać przy obronie. Puszczaj po sekcjach 3–5, jeśli zostanie czas.

| test | run | pytanie | porównanie z | czas (L4) |
|---|---|---|---|---|
| 6a | zwykły U-Net, `--lr 1e-4` i `--lr 1e-3`, 30 epok | czy U-Net miał zły współczynnik uczenia? | `*_unet_plain` z 06 (lr 3e-4) | 2 × ~11 min |
| 6b | `corr --width 64` (~4× więcej wag, ok. 25 M), 80 epok | czy sieć bez geometrii jest po prostu za mała? | `*_corr` z 08 (width 32, 6,2 M) | ~2 h |

Jak czytać: jeśli 6a mieści się w ok. 0,5 dB, wynik nie zależy od lr; jeśli 6b też stoi w okolicy 30 dB, granica wynika
z architektury (braku geometrii), a nie z liczby wag. Ważniejszy jest test 6b.

In [ ]:
CONTROL = {   # nazwa runu: (tryb, epoki, lr, szerokość)
    f'{DAY}_unet_plain_lr1e-4': ('plain', 30, 1e-4, 32),
    f'{DAY}_unet_plain_lr1e-3': ('plain', 30, 1e-3, 32),
    f'{DAY}_corr_w64':          ('corr', 80, 3e-4, 64),
}

def train_control(run, resume=False):
    mode, epochs, lr, width = CONTROL[run]
    res = f'--resume "{P.RUNS / run / "last.pt"}"' if resume else ''
    !python src/train_unet.py --data "{DATA}" --mode {mode} --width {width} --epochs {epochs} --bs {BS} --lr {lr} --workers {WORKERS} --out "{P.RUNS}" --name {run} {res}
    prev = sorted((P.RUNS / run).glob('preview_ep*.png'))
    if prev:
        print(prev[-1]); display(Image(str(prev[-1])))
print(list(CONTROL))

### 6a. Współczynnik uczenia zwykłego U-Netu (po rozłączeniu: `train_control(..., resume=True)`)

In [ ]:
train_control(f'{DAY}_unet_plain_lr1e-4')

In [ ]:
train_control(f'{DAY}_unet_plain_lr1e-3')

### 6b. Szersza sieć `corr` (width 64)

In [ ]:
train_control(f'{DAY}_corr_w64')

## 7. Porównanie
Modele, z którymi porównujemy, są brane z `results.json` runów z 04, 06 i 08; testy kontrolne pokazują się, jeśli były uruchomione (najnowszy run o danej nazwie).

In [ ]:
import json, pandas as pd
def metrics(rj):
    m = json.load(open(rj))
    return m.get('val_metrics') or m.get('metrics') or m
rows = []
def add(question, label, rj):
    if rj and rj.exists():
        m = metrics(rj)
        rows.append({'pytanie': question, 'model': label, 'run': rj.parent.name, 'psnr': m.get('psnr'),
                     'psnr_full': m.get('psnr_full'), 'ssim': m.get('ssim'), 'wejście (full)': m.get('psnr_full_in')})
    else:
        print('brak', label, rj)
add('A: kamera B', 'zwykły U-Net, A + B, 30 ep.', P.find_run('unet_plain', 'results.json'))
add('A: kamera B', 'zwykły U-Net, tylko A, 30 ep.', P.RUNS / RUNS['plain'] / 'results.json')
add('A: kamera B', 'corr, A + B, 80 ep.', P.find_run('corr', 'results.json'))
add('A: kamera B', 'corr, tylko A, 80 ep.', P.RUNS / RUNS['corr'] / 'results.json')
add('B: geometria', 'corr (bez geometrii), 200 ep.', P.find_run('corr200', 'results.json'))
add('B: geometria', 'korelacja + geometria (od zera), 200 ep.', P.RUNS / RUNS['corrgeom'] / 'results.json')
add('B: geometria', 'RAFT gotowy + geometria v1 (04)', P.find_run('raft_zeroshot', 'eval.json'))
add('B: geometria', 'RAFT dostrojony + geometria v1 (04)', P.find_run('raft_geom', 'results.json'))
for run, (mode, epochs, lr, width) in CONTROL.items():
    rj = P.RUNS / run / 'results.json'
    if rj.exists():
        add('C: kontrola', f'{mode}, lr {lr:g}, width {width}, {epochs} ep.', rj)
add('C: kontrola', 'plain, lr 0.0003, width 32, 30 ep. (06)', P.find_run('unet_plain', 'results.json'))
add('C: kontrola', 'corr, lr 0.0003, width 32, 80 ep. (08)', P.find_run('corr', 'results.json'))
display(pd.DataFrame(rows).set_index(['pytanie', 'model']).round(2))
print('val rs_video_x4k (321 próbek); psnr = obszar widoczny w A, psnr_full = cały obraz (dla RAFT z 04 tylko psnr)')